# Inspect `cusip_ranks_v2__change_in_weight.parquet`

Quick description of the per-stock per-quarter rank parquet produced by `sweep_features_v2.py`.

In [ ]:
from pathlib import Path
import pandas as pd

PATH = Path('cusip_ranks_v2__change_in_weight.parquet')
df = pd.read_parquet(PATH)
print(f'file:   {PATH}')
print(f'shape:  {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'memory: {df.memory_usage(deep=True).sum() / 1e6:.2f} MB')

## Schema (column types)

In [ ]:
df.dtypes

## Head (first 10 rows)

In [ ]:
df.head(10)

## Numeric summary

In [ ]:
df.describe(include='all').T

## Coverage: how many quarters and stocks

In [ ]:
quarters = df.groupby(['year', 'quarter']).size().rename('n_stocks').reset_index()
print(f"unique quarters:    {len(quarters)}")
print(f"range:              {int(quarters['year'].min())}Q{int(quarters.iloc[0]['quarter'])} "
      f"-> {int(quarters['year'].max())}Q{int(quarters.iloc[-1]['quarter'])}")
print(f"unique CUSIPs:      {df['cusip'].nunique():,}")
if 'model' in df.columns:
    print(f"unique model tags:  {df['model'].unique().tolist()}")
quarters.head(15)

## Top 10 stocks for the most recent quarter

In [ ]:
last_y, last_q = df.sort_values(['year', 'quarter']).iloc[-1][['year', 'quarter']]
last_y, last_q = int(last_y), int(last_q)
print(f'top 10 stocks for {last_y}Q{last_q}:')
(df[(df['year'] == last_y) & (df['quarter'] == last_q)]
   .sort_values('rank').head(10))

## What each column means

| Column | Meaning |
|---|---|
| `cusip` | Stock identifier |
| `year`, `quarter` | The quarter the model was TRAINED on |
| `mean_score` | Mean of `σ(fund_emb · stock_emb)` across all funds in that quarter — a stock's "average attractiveness" to the cohort of funds. Higher = more funds tend to touch this stock per the model |
| `rank` | 1 = highest `mean_score` in that quarter, len(stocks) = lowest |
| `model` | Tag identifying which sweep produced the row (e.g. `WeightedLightGCN_v2`) |

Note: in v2 these scores reflect Q's edges only (no out-of-sample prediction — that arrived in v3).